# Solvating your protein in a water box

This assumes you have already processed your protein pdb file using the previous script, and have a file of the form `{id}_fixer.pdb`

Goals for this notebook.
1. Add any missing hydrogens
2. Imerse the protein in a solvent box with water and ions
3. Visualize and confirm the result

 

In [ ]:
from pathlib import Path

import openmm as mm
import openmm.app as app
import openmm.unit as unit

import py3Dmol

from sys import stdout


## Setup Protein in a water box

To mimic the cellular environment, we usually immerse the protein in a box of water molecules. The box uses periodic boundary conditions to simulate an extended expanse of water.  When an atom passes through the box boundary on one side, an identical copy enters the box on the opposite side with the same velocity.  While only the central box is updated during a simulation, the neighbouring boxes are included in the calculation of long range forces (coulombic electrostatic forces). One requirement for using periodic boundary conditions is the solvent box must be charge neutral.  So if the protein had a +5 overall charge, an extra 5 $Cl^-$ ions would be added to ensure neutrality. 

If you have ligands or non-standard residues, the setup is more complicated.  Master the protein setup first before moving on to those cases.


In [ ]:
id = '4trn'
Path('pdbs').mkdir(exist_ok=True)

# AMBER19 protein force field and TIP3P-FB water model 
forcefield = app.ForceField('amber19-all.xml', 'amber19/tip3pfb.xml')

print("Loading protein and adding hydrogens...")
pdb = app.PDBFile(f"./pdbs/{id}_fixer.pdb")
modeller = app.Modeller(pdb.topology, pdb.positions)

# 
modeller.addHydrogens(forcefield, pH=7.0)

print("Adding a solvent box...")
modeller.addSolvent(
    forcefield,
    padding= 1.0 * unit.nanometers,      # minimum solvent padding around protein
    ionicStrength= 0.15 * unit.molar,    # typical ionic strength for cellular environments
    neutralize=True 
)

print(f"Saving solvated system... check {id}_solvated.pdb")
with open(f"./pdbs/{id}_solvated.pdb", "w") as f:
    app.PDBFile.writeFile(
        modeller.topology,
        modeller.positions,
        f
    )


In [ ]:
with open(f"./pdbs/{id}_solvated.pdb") as f:
    pdb_string = f.read()

view = py3Dmol.view(width=900, height=600)
view.addModel(pdb_string, "pdb")

# Protein as cartoon
view.setStyle({"chain": "A"}, {"cartoon": {"color": "spectrum"}})
# Ligands as sticks
view.setStyle({"hetflag": True, "not": {"resn": ["HOH", "WAT", "NA", "CL"]}}, {"stick": {}})
# Waters as smaller sticks
view.setStyle({"resn": ["HOH", "WAT"]},{"sphere": {"radius":0.2}})
# Ions as small spheres
view.setStyle({"resn": ["NA", "CL", "K", "CA", "MG"]},{"sphere": {"radius": 0.4}})
view.zoomTo()
view.show()